# Avant le TP8 : la boîte à outils du diagnostic

Au TP8, on ne vous dit pas ce qui est cassé : un ticket décrit un symptôme, et vous enquêtez. Cette démonstration montre, sur le lab en bon état, les quatre réflexes de la boîte à outils :

1. un **bilan** général, du quorum aux alertes, qui sert de référence ;
2. la **comparaison** d'une configuration avec sa version d'origine ;
3. la recherche dans les **traces** d'un nœud ;
4. les **alertes** et une métrique de Prometheus.

Sorties de référence : exécution du 07/11/2026 22:41 UTC sur le lab (mode `extended`).

In [1]:
import os
import subprocess
import time

import requests
from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]
PROMETHEUS = "http://prometheus:9090"
admin = AdminClient({"bootstrap.servers": BS})

## 1. Le bilan de référence

Une seule fonction rassemble l'état général : le leader du quorum, le nom sous lequel chaque broker est annoncé aux clients, les partitions dégradées et les alertes. Exécutée sur un lab sain, elle donne la référence « tout va bien ». Au TP, vous y ajouterez un indicateur.

In [2]:
def alertes():
    donnees = requests.get(PROMETHEUS + "/api/v1/alerts", timeout=10).json()["data"]["alerts"]
    return sorted({"%s:%s" % (a["labels"]["alertname"], a["state"]) for a in donnees})


def bilan():
    statut = subprocess.run(["docker", "exec", "kafka-tools", "kafka-metadata-quorum.sh",
                             "--bootstrap-server", BS, "describe", "--status"],
                            capture_output=True, text=True).stdout
    partitions = [p for t in admin.list_topics(timeout=10).topics.values() for p in t.partitions.values()]
    print("quorum         leader", statut.split("LeaderId:")[1].split()[0])
    print("brokers       ", {n.id: n.host for n in sorted(admin.describe_cluster().result().nodes, key=lambda n: n.id)})
    print("urp           ", sum(1 for p in partitions if len(p.isrs) < len(p.replicas)))
    print("indisponibles ", sum(1 for p in partitions if p.leader == -1))
    print("alertes       ", alertes())


bilan()

quorum         leader 3
brokers        {1: 'kafka-1', 2: 'kafka-2', 3: 'kafka-3'}
urp            0
indisponibles  0
alertes        []


## 2. Qu'est-ce qui a changé ?

Le lab garde la configuration d'origine de chaque nœud dans `config/kafka/pristine/`. Une différence avec le fichier en service répond à la question « qu'est-ce qui a changé ? ». Ici, `kafka-1` : s'il a gardé le second répertoire de données du TP7, `diff` le montre ; sinon, la sortie est vide.

In [3]:
%%bash
cd $FORMATION_DIR/lab
diff config/kafka/pristine/kafka-1.server.properties config/kafka/kafka-1/server.properties

## 3. Chercher dans les traces d'un nœud

`docker compose logs` donne les traces d'un nœud ; `grep -E` les filtre. On compte d'abord les avertissements et les erreurs des 10 dernières minutes, puis on lit les dernières lignes.

In [4]:
%%bash
cd $FORMATION_DIR/lab
for n in 1 2 3; do
  echo "kafka-$n : $(docker compose logs kafka-$n --since 10m --no-log-prefix 2>&1 | grep -cE ' (WARN|ERROR) ') ligne(s) WARN ou ERROR en 10 min"
done

kafka-1 : 6 ligne(s) WARN ou ERROR en 10 min


kafka-2 : 3 ligne(s) WARN ou ERROR en 10 min


kafka-3 : 7 ligne(s) WARN ou ERROR en 10 min


In [5]:
%%bash
cd $FORMATION_DIR/lab
docker compose logs kafka-1 --no-log-prefix 2>&1 | grep -E ' (INFO|WARN) ' | tail -3 | cut -c1-200

[2026-11-07 22:42:16,176] INFO [GroupCoordinator id=1 topic=__consumer_offsets partition=33] Generat

ed 0 records in 0 milliseconds while cleaning share group state for topics [Z8mY1lJXTRqwknOiRstriQ].


[2026-11-07 22:42:16,176] INFO [GroupCoordinator id=1 topic=__consumer_offsets partition=9] Generat

ed 0 records in 0 milliseconds while cleaning share group state for topics [Z8mY1lJXTRqwknOiRstriQ].

 
[2026-11-07 22:42:16,176] INFO [GroupCoordinator id=1 topic=__consumer_offsets partition=31] Gener

ated 0 records in 0 milliseconds while cleaning share group state for topics [Z8mY1lJXTRqwknOiRstriQ

].


## 4. Alertes et métriques

Les alertes disent **que** quelque chose ne va pas ; les métriques aident à trouver **où**. Exemple : le débit d'entrée par topic.

In [6]:
reponse = requests.get(PROMETHEUS + "/api/v1/query", timeout=10, params={
    "query": 'sum by (topic) (rate(kafka_server_brokertopicmetrics_messagesin_total{topic!=""}[1m]))'}).json()
for resultat in reponse["data"]["result"]:
    print("%-20s %6.1f messages/s" % (resultat["metric"]["topic"], float(resultat["value"][1])))

loadgen                97.5 messages/s
__consumer_offsets      1.3 messages/s


## La méthode du TP8

Ouvrez `tps/jour2/tp8_incidents/enonce.ipynb`, et dupliquez `fiche-diagnostic.md` une fois par ticket.

- Qualifier, observer du général au particulier, formuler une hypothèse, la vérifier par une commande, corriger, valider. Une cellule par hypothèse.
- Observez avant d'agir : un redémarrage efface des indices.
- Votre partenaire injecte l'incident dans un terminal de votre poste ; il joue l'équipe qui a ouvert le ticket et répond sur le contexte, jamais sur la cause.